In [1]:
%load_ext autoreload

```python
from base64 import b64decode

import matplotlib.pyplot as plt
import numpy as np
from IPython import display
from selenium.webdriver.common.by import By
from sklearn.cluster import DBSCAN
```


In [3]:
%autoreload 2
from mabot.browser.browser import AgentBrowser
from mabot.browser.markup import WebElementSelectorByCursorCss

In [4]:
selector = WebElementSelectorByCursorCss(
    style=["pointer", "text"], is_displayed=True
)
browser = AgentBrowser(
    "https://stackoverflow.com/questions/25019752/select-an-element-by-the-text-it-contains-with-selenium-webdriver/25020114#25020114",
    selector=selector,
)

2025-01-07 10:43:16.122 | INFO     | mabot.browser.browser:__init__:36 - Initialize driver
2025-01-07 10:43:17.838 | INFO     | mabot.browser.browser:__init__:38 - Get URL: 'https://stackoverflow.com/questions/25019752/select-an-element-by-the-text-it-contains-with-selenium-webdriver/25020114#25020114'
2025-01-07 10:43:38.763 | INFO     | mabot.browser.markup:__init__:132 - Number of elements: 190
2025-01-07 10:43:38.766 | INFO     | mabot.browser.browser:__init__:43 - Initialize browser. Start from 'https://stackoverflow.com/questions/25019752/select-an-element-by-the-text-it-contains-with-selenium-webdriver/25020114#25020114'


In [5]:
browser.driver

<selenium.webdriver.chrome.webdriver.WebDriver (session="9d05371ee0f96219307c7213d483e7af")>

```python
import networkx as nx

G = nx.Graph()

root = browser.driver.find_elements(by=By.XPATH, value="//body")[0]
level = [root]
G.add_node(root.id)
width = 0
height = 0
while level:
    next_level = []

    for tag in level:
        if not tag.is_displayed():
            continue
        children = tag.find_elements(by=By.XPATH, value="./*")
        if len(children) > 0:
            for child in children:
                G.add_node(child.id)
                G.add_edge(tag.id, child.id)
            next_level += children

    height += 1
    level = next_level
```


```python
root.id in list(G.nodes)
```


```python
pos = nx.bfs_layout(G, root.id)
nx.draw(G, pos)
plt.show()
```


pos

```python
import matplotlib.pyplot as plt
import networkx as nx
from networkx.drawing.nx_pydot import graphviz_layout

pos = graphviz_layout(G, prog="dot")
nx.draw(G, pos)
plt.show()
```


```python
len(
    nx.shortest_path(
        G, browser.markup._elements[0].id, browser.markup._elements[1].id
    )
)
```


```python
dist = np.zeros((len(browser.markup._elements), len(browser.markup._elements)))
for i in browser.markup._elements:
    for j in browser.markup._elements:
        dist[i, j] = len(
            nx.shortest_path(
                G,
                browser.markup._elements[i].id,
                browser.markup._elements[j].id,
            )
        )
```


<a href="https://twitter.com/stackoverflow" class="-link js-gps-track" data-gps-track="footer.click({ location: 2, link: 32 })">Twitter</a>


```python
cluster = DBSCAN(min_samples=2, eps=0.1)
# kmeans = KMeans(n_clusters=5, random_state=22)
```


```python
data = np.empty((len(browser.markup._elements), 4))
for i, element in browser.markup._elements.items():
    location = element.location
    data[i, 0] = pos[element.id][0]
    data[i, 1] = pos[element.id][1]
    data[i, 2] = location["x"]
    data[i, 3] = location["y"]

data /= data.max(axis=0)
```


```python
result = cluster.fit(data)
np.unique(result.labels_)
```


```python
list(enumerate(result.labels_))
```


```python
view = browser.get_view_base64()
display.Image(b64decode(view))
```
